# Diabetes Prediction using Machine Learning


# New Section

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
file_path='/content/drive/MyDrive/dataset/diabetes.csv'
data=pd.read_csv(file_path)

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data.size

In [ ]:
data.isna()

In [ ]:
# Bar chart
sns.countplot(x='Outcome', data=data, palette='pastel')
plt.title("Class Distribution (0=Non-Diabetic, 1=Diabetic)")
plt.show()



In [ ]:
features = data.columns[:-1]  # all except Outcome
for col in features:
    plt.figure(figsize=(6,4))
    sns.histplot(data[data['Outcome']==0][col], label="Non-Diabetic", color="skyblue", kde=True, stat="density")
    sns.histplot(data[data['Outcome']==1][col], label="Diabetic", color="salmon", kde=True, stat="density")
    plt.title(f"Distribution of {col}")
    plt.legend()
    plt.show()


In [ ]:
for col in features:
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Outcome", y=col, data=data, palette="pastel")
    plt.title(f"{col} vs Outcome")
    plt.show()

    plt.figure(figsize=(6,4))
    sns.violinplot(x="Outcome", y=col, data=data, palette="muted", split=True)
    plt.title(f"{col} vs Outcome (Violin)")
    plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x="Glucose", y="BMI", hue="Outcome", data=data, palette=['skyblue','salmon'])
plt.title("BMI vs Glucose by Outcome")
plt.show()


In [ ]:
sns.pairplot(data[['Glucose','BMI','Age','Insulin','Outcome']], hue="Outcome", palette=['skyblue','salmon'])
plt.suptitle("Pairplot of Selected Features", y=1.02)
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
corr = data.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = data.drop("Outcome", axis=1)
y = data["Outcome"]

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances, y=importances.index, palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)   # use scaled data

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:,1]

print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)   # no scaling needed

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

svm = SVC(probability=True, kernel="rbf", random_state=42)
svm.fit(X_train_scaled, y_train)   # scaling is very important for SVM

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:,1]

print("SVM")
print(classification_report(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_svm))


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)   # scaling is crucial for KNN

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:,1]

print("KNN")
print(classification_report(y_test, y_pred_knn))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_knn))


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-Diabetic","Diabetic"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
pickle.dump(log_reg, open("DiabetesPrediction.pkl", "wb"))

In [ ]:
!ls

In [ ]:
from google.colab import files
files.download("DiabetesPrediction.pkl")